# FoML Project Workflow Notebook

This notebook consolidates the repository workflow into one place.

It covers:
- environment setup
- dataset loading and quick inspection
- cached feature and graph generation
- EDA generation
- model training
- hyperparameter tuning
- ablation studies
- output inspection

The underlying project scripts are reused directly, so the notebook stays aligned with the codebase.

In [45]:
import torch

In [47]:
torch.cuda.is_available()

False

In [ ]:
!ssh node1

Last login: Fri Apr 17 09:11:15 2026 from 192.168.10.254


In [39]:
import json
import os
import runpy
import sys
from pathlib import Path
import torch
MIN_PYTHON = (3, 10)
if sys.version_info < MIN_PYTHON:
    required = '.'.join(map(str, MIN_PYTHON))
    current = '.'.join(map(str, sys.version_info[:3]))
    raise RuntimeError(
        f"This notebook requires Python {required}+ because the project uses modern type-hint syntax. "
        f"Current kernel: {current}. Switch the Jupyter kernel to a newer interpreter and rerun."
    )

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "README.md").exists():
    raise RuntimeError("Open this notebook from the project root directory.")

for path in (PROJECT_ROOT, PROJECT_ROOT.parent.parent):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

Project root: /home/herrys/projects/FoML_Project
Python: /home/herrys/miniconda3/bin/python


In [40]:
import json
import runpy
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()

from utils.data_utils import load_dataset
from utils.project_paths import DATA_DIR, OUTPUTS_DIR


def run_script(relative_path: str, *args: object) -> dict[str, object]:
    script_path = PROJECT_ROOT / relative_path
    argv_backup = sys.argv[:]
    sys.argv = [str(script_path), *[str(arg) for arg in args]]
    try:
        print(f"Running {relative_path} {' '.join(map(str, args))}".strip())
        return runpy.run_path(str(script_path), run_name="__main__")
    finally:
        sys.argv = argv_backup


def collect_metrics() -> pd.DataFrame:
    rows = []
    for metrics_path in sorted(OUTPUTS_DIR.glob("*/*/metrics.json")):
        payload = json.loads(metrics_path.read_text(encoding="utf-8"))
        rows.append(
            {
                "family": metrics_path.parent.parent.name,
                "model_name": metrics_path.parent.name,
                **payload,
            }
        )
    if not rows:
        return pd.DataFrame(columns=["family", "model_name", "rmse", "mae", "r2"])
    return pd.DataFrame(rows).sort_values(by="rmse", ascending=True).reset_index(drop=True)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE = device
RUN_CACHE_BUILDS = True
RUN_EDA = True
RUN_TRAINING = True
RUN_FULL_PIPELINE = True
RUN_TUNING = True
RUN_ABLATION = True

display(Markdown(
    f"""
### Runtime Flags
- `DEVICE = {DEVICE}`
- `RUN_CACHE_BUILDS = {RUN_CACHE_BUILDS}`
- `RUN_EDA = {RUN_EDA}`
- `RUN_TRAINING = {RUN_TRAINING}`
- `RUN_FULL_PIPELINE = {RUN_FULL_PIPELINE}`
- `RUN_TUNING = {RUN_TUNING}`
- `RUN_ABLATION = {RUN_ABLATION}`
"""
))


### Runtime Flags
- `DEVICE = cpu`
- `RUN_CACHE_BUILDS = True`
- `RUN_EDA = True`
- `RUN_TRAINING = True`
- `RUN_FULL_PIPELINE = True`
- `RUN_TUNING = True`
- `RUN_ABLATION = True`


## Dataset Preview

In [41]:
frame = load_dataset()
print(frame.shape)
frame[["Name", "SMILES", "Solubility"]].head(10)

[16:05:38] Explicit valence for atom # 5 N, 4, is greater than permitted
[16:05:38] Explicit valence for atom # 5 N, 4, is greater than permitted


(9980, 27)


,Name,SMILES,Solubility
0,"N,N,N-trimethyloctadecan-1-aminium bromide",[Br-].CCCCCCCCCCCCCCCCCC[N+](C)(C)C,-3.616127
1,Benzo[cd]indol-2(1H)-one,O=C1Nc2cccc3cccc1c23,-3.254767
2,4-chlorobenzaldehyde,Clc1ccc(C=O)cc1,-2.177078
3,"zinc bis[2-hydroxy-3,5-bis(1-phenylethyl)benzo...",[Zn++].CC(c1ccccc1)c2cc(C(C)c3ccccc3)c(O)c(c2)...,-3.924409
4,4-({4-[bis(oxiran-2-ylmethyl)amino]phenyl}meth...,C1OC1CN(CC2CO2)c3ccc(Cc4ccc(cc4)N(CC5CO5)CC6CO...,-4.662065
5,vinyltoluene,Cc1cccc(C=C)c1,-3.123150
6,3-(3-ethylcyclopentyl)propanoic acid,CCC1CCC(CCC(O)=O)C1,-3.286116
7,"11,16,17,21-tetrahydroxypregna-1,4-diene-3,20-...",CC12CC(O)C3C(CCC4=CC(=O)C=CC34C)C1CC(O)C2(O)C(...,-2.664549
8,bis(4-fluorophenyl)methanone,Fc1ccc(cc1)C(=O)c2ccc(F)cc2,-4.396652
9,1-[2-(benzoyloxy)propoxy]propan-2-yl benzoate ...,O=C(OCCCOCCCOC(=O)c1ccccc1)c2ccccc2,-4.595503


## Cached Data Generation

Set `RUN_CACHE_BUILDS = True` above to regenerate cached tabular and graph inputs.

In [42]:
if RUN_CACHE_BUILDS:
    run_script("utils/make_data.py")
    run_script("utils/make_graph_data.py")
else:
    print("Skipping cache generation.")

Running utils/make_data.py


[16:05:40] Explicit valence for atom # 5 N, 4, is greater than permitted
[16:05:40] Explicit valence for atom # 5 N, 4, is greater than permitted


Saved classical features with shape (9980, 1036) and targets with shape (9980,).
Running utils/make_graph_data.py


[16:05:56] Explicit valence for atom # 5 N, 4, is greater than permitted
[16:05:56] Explicit valence for atom # 5 N, 4, is greater than permitted


Saved graph dataset with 9980 molecules.


## Exploratory Data Analysis

Set `RUN_EDA = True` to regenerate the report-ready EDA assets in `outputs/eda/`.

In [43]:
if RUN_EDA:
    run_script("EDA/generate_eda_report.py")
else:
    print("Skipping EDA generation.")

Running EDA/generate_eda_report.py


[16:06:14] Explicit valence for atom # 5 N, 4, is greater than permitted
[16:06:14] Explicit valence for atom # 5 N, 4, is greater than permitted


Saved EDA outputs to /home/herrys/projects/FoML_Project/outputs/eda


## Training

Set `RUN_TRAINING = True` to run each model script individually, or `RUN_FULL_PIPELINE = True` to run the combined pipeline.

In [44]:
training_scripts = [
    "train/train_linear_regression.py",
    "train/train_gaussian_process.py",
    "train/train_mlp_regressor.py",
    "train/train_dense_regressor.py",
    "train/train_graph_cn.py",
    "train/train_graph_net.py",
    "train/train_graph_sage.py",
    "train/train_graph_mp.py",
]

if RUN_TRAINING:
    for script in training_scripts[:3]:
        run_script(script)
    for script in training_scripts[3:]:
        run_script(script, "--device", DEVICE)
else:
    print("Skipping per-model training.")

if RUN_FULL_PIPELINE:
    run_script("train/run_full_pipeline.py", "--device", DEVICE)
else:
    print("Skipping full pipeline run.")

Running train/train_linear_regression.py


[16:06:18] Explicit valence for atom # 5 N, 4, is greater than permitted
[16:06:18] Explicit valence for atom # 5 N, 4, is greater than permitted


{
  "rmse": 1.5715147256851196,
  "mae": 1.191395878791809,
  "r2": 0.5444765686988831,
  "model_variant": "linear",
  "feature_mode": "combined",
  "source_feature_mode": "combined",
  "num_features": 1036,
  "pca_explained_variance": null,
  "alpha": 1.0,
  "l1_ratio": 0.5,
  "fit_intercept": true,
  "positive": false,
  "train_rows": 7984,
  "test_rows": 1996
}
Running train/train_gaussian_process.py


[16:06:34] Explicit valence for atom # 5 N, 4, is greater than permitted
[16:06:34] Explicit valence for atom # 5 N, 4, is greater than permitted
/home/herrys/miniconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


{
  "rmse": 2.05270258492531,
  "mae": 1.5556997422018204,
  "r2": 0.22281226632654416,
  "feature_mode": "descriptor",
  "source_feature_mode": "descriptor",
  "num_features": 12,
  "pca_explained_variance": null,
  "kernel_name": "rbf",
  "length_scale": 1.0,
  "matern_nu": 1.5,
  "rq_alpha": 1.0,
  "alpha": 1e-06,
  "train_rows": 7984,
  "test_rows": 1996
}
Running train/train_mlp_regressor.py


[16:08:12] Explicit valence for atom # 5 N, 4, is greater than permitted
[16:08:12] Explicit valence for atom # 5 N, 4, is greater than permitted


{
  "rmse": 1.2087767124176025,
  "mae": 0.8517211675643921,
  "r2": 0.7304954528808594,
  "feature_mode": "combined",
  "source_feature_mode": "combined",
  "num_features": 1036,
  "pca_explained_variance": null,
  "hidden_layers": [
    256,
    128,
    64
  ],
  "alpha": 0.0001,
  "learning_rate_init": 0.001,
  "train_rows": 7984,
  "test_rows": 1996
}
Running train/train_dense_regressor.py --device cpu


[16:09:00] Explicit valence for atom # 5 N, 4, is greater than permitted
[16:09:00] Explicit valence for atom # 5 N, 4, is greater than permitted


KeyboardInterrupt: 

## Hyperparameter Tuning

Set `RUN_TUNING = True` to execute classical, tabular DNN, and graph tuning scripts.

In [ ]:
if RUN_TUNING:
    run_script("tuning/tune_classical_models.py")
    run_script("tuning/tune_dnn.py", "--device", DEVICE)
    run_script("tuning/tune_graph_models.py", "--device", DEVICE)
else:
    print("Skipping tuning.")

Skipping tuning.


## Ablation Studies

Set `RUN_ABLATION = True` to run both binary-classification ablation scripts.

In [ ]:
if RUN_ABLATION:
    run_script("ablation/run_feature_ablation.py")
    run_script("ablation/run_graph_ablation.py")
else:
    print("Skipping ablation studies.")

Skipping ablation studies.


## Outputs Summary

In [ ]:
metrics_table = collect_metrics()
print(f"Outputs directory: {OUTPUTS_DIR}")
print(f"Metrics files found: {len(metrics_table)}")
metrics_table

Outputs directory: /home/herrys/projects/FoML_Project/outputs
Metrics files found: 14


,family,model_name,rmse,mae,r2,feature_mode,source_feature_mode,num_features,pca_explained_variance,kernel_name,...,learning_rate_init,dropout,epochs,batch_size,learning_rate,weight_decay,device,hidden_channels,train_graphs,test_graphs
0,graphml,graph_sage,1.140104,0.816732,0.760248,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,40.0,16.0,0.001,0.00001,NaN,64.0,7984.0,1996.0
1,dnn,dense_regressor,1.177536,0.835794,0.744246,combined,combined,1036.0,NaN,NaN,...,NaN,0.15,150.0,64.0,0.001,0.00001,cuda,NaN,NaN,NaN
2,dnn,sklearn_mlp_regressor,1.208777,0.851721,0.730495,combined,combined,1036.0,NaN,NaN,...,0.001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,classical,mlp_regressor,1.208777,0.851721,0.730495,combined,NaN,1036.0,NaN,NaN,...,0.001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,classical,ridge_regression,1.430800,1.039603,0.622400,combined,NaN,1036.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,classical,linear_regression,1.571513,1.191395,0.544478,combined,combined,1036.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,dnn,sklearn_mlp_regressor_pca3d,1.854179,1.439242,0.365872,pca3d,combined,3.0,0.036253,NaN,...,0.001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,dnn,dense_regressor_pca3d,1.889421,1.472706,0.341537,pca3d,combined,3.0,0.036198,NaN,...,NaN,0.15,150.0,64.0,0.001,0.00001,cuda,NaN,NaN,NaN
8,classical,gaussian_process,2.052703,1.555700,0.222812,descriptor,descriptor,12.0,NaN,rbf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,graphml,graph_mp,2.170841,1.735770,0.130779,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,120.0,32.0,0.001,0.00001,NaN,64.0,7984.0,1996.0


In [ ]:
csv_files = sorted(OUTPUTS_DIR.rglob("*.csv"))
pd.DataFrame(
    {
        "relative_path": [str(path.relative_to(PROJECT_ROOT)) for path in csv_files],
        "size_bytes": [path.stat().st_size for path in csv_files],
    }
)

,relative_path,size_bytes
0,outputs/classical/gaussian_process/predictions...,964995
1,outputs/classical/gaussian_process_pca3d/predi...,964036
2,outputs/classical/linear_regression/prediction...,964411
3,outputs/classical/linear_regression_pca3d/pred...,963626
4,outputs/classical/mlp_regressor/predictions.csv,965213
5,outputs/classical/ridge_regression/predictions...,964753
6,outputs/dnn/dense_regressor/history.csv,2736
7,outputs/dnn/dense_regressor/predictions.csv,965162
8,outputs/dnn/dense_regressor_pca3d/history.csv,2205
9,outputs/dnn/dense_regressor_pca3d/predictions.csv,964021


## Streamlit Dashboard

Launch the app separately from a terminal if needed:

```bash
streamlit run app.py
```

streamlit run app.py

In [ ]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.64.18.52:8501
  External URL: http://14.139.128.20:8501

  Stopping...
^C
  Stopping...
Exception ignored on threading shutdown:
Traceback (most recent call last):
  File "/home/herrys/miniconda3/lib/python3.13/threading.py", line 1537, in _shutdown
    atexit_call()
  File "/home/herrys/miniconda3/lib/python3.13/threading.py", line 1508, in <lambda>
    _threading_atexits.append(lambda: func(*arg, **kwargs))
  File "/home/herrys/miniconda3/lib/python3.13/concurrent/futures/thread.py", line 31, in _python_exit
    t.join()
  File "/home/herrys/miniconda3/lib/python3.13/threading.py", line 1095, in join
    self._handle.join(timeout)
  File "/home/herrys/miniconda3/lib/python3.13/site-packages/streamlit/web/bootstrap.py", line 43, in signal_handler
    server.stop()
  File "/home/herrys/miniconda3/lib/python3.13/site-packages/streamlit/web/server/server.py", line 550, i